#### testing weather files

In [ ]:
import pandas as pd

def epw_to_df(epw_path):
    with open(epw_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    data_start = next(i for i, line in enumerate(lines) if line.startswith('DATA PERIODS')) + 1
    header = ['Year','Month','Day','Hour','Minute','DataSource','DryBulbTemp','DewPointTemp','RelativeHumidity',
              'AtmosphericStationPressure','ExtraterrestrialHorizRad','ExtraterrestrialDirNormRad','HorizontalInfraredRadIntSky',
              'GlobalHorizontalRadiation','DirectNormalRadiation','DiffuseHorizontalRadiation','GlobalHorizontalIlluminance',
              'DirectNormalIlluminance','DiffuseHorizontalIlluminance','ZenithLuminance','WindDirection','WindSpeed',
              'TotalSkyCover','OpaqueSkyCover','Visibility','CeilingHeight','PresentWeatherObservation','PresentWeatherCodes',
              'PrecipitableWater','AerosolOpticalDepth','SnowDepth','DaysSinceLastSnowfall','Albedo','LiquidPrecipitationDepth',
              'LiquidPrecipitationQuantity']
    df = pd.DataFrame([l.strip().split(',') for l in lines[data_start:]], columns=header)
    df = df.apply(pd.to_numeric, errors='ignore')
    df.loc[df['Hour'] == 24, 'Hour'] = 0
    df['Datetime'] = pd.to_datetime(df[['Year','Month','Day','Hour']])
    return df

weather_files = {
    'Beizaee_Oiko': "/workspaces/CUBES/cubes/data/weather/loughborough_beizaee_oiko.epw",
    'Beizaee': "/workspaces/CUBES/cubes/data/weather/loughborough_beizaee.epw",
    'Standard': "/workspaces/CUBES/cubes/data/weather/loughborough.epw"
}

metrics = {'Temperature':'DryBulbTemp','Solar':'GlobalHorizontalRadiation','Wind':'WindSpeed'}
dates = {
    'Whole28Days': ('2014-02-16','2014-03-15'),
    'Thursday': ('2014-02-27','2014-02-27'),
    'Friday': ('2014-02-28','2014-02-28'),
    'Saturday': ('2014-03-01','2014-03-01')
}

def avg_metrics(d): return {k:d[v].mean() for k,v in metrics.items()}

all_results=[]
for label,path in weather_files.items():
    df=epw_to_df(path)
    results={'WeatherFile':label}
    for period,(s,e) in dates.items():
        subset=df[(df['Datetime']>=s)&(df['Datetime']<=e)]
        for k,v in avg_metrics(subset).items():
            results[f'{period}_{k}']=v
    all_results.append(results)

comparison=pd.DataFrame(all_results).set_index('WeatherFile')
print(comparison.round(2))

In [1]:
import pandas as pd

html_path = "/workspaces/CUBES/beizaee_validation/runs_h28_test_validation/beizaee/conventional_control__part_0_2__eff_quadratic/eplustbl.htm"

# Read all tables, keeping headers as None so we control them
tables = pd.read_html(html_path, header=None)

zone_info = None
for df in tables:
    # Look for a row that contains "Zone Name" → treat that as header
    header_row = df[df.astype(str).apply(lambda row: row.str.contains("Zone Name", na=False)).any(axis=1)]
    if not header_row.empty:
        # Use that row as header, reset the DataFrame
        header_idx = header_row.index[0]
        df.columns = df.iloc[header_idx]
        df = df.drop(index=range(0, header_idx+1))  # drop header rows above
        df = df.reset_index(drop=True)
        zone_info = df
        break

if zone_info is not None:
    # Convert numeric columns
    for col in zone_info.columns:
        zone_info[col] = pd.to_numeric(zone_info[col], errors="ignore")

    print(zone_info.head())
    print(f"Total floor area = {zone_info['Floor Area {m2}'].sum():.2f} m²")
    print(f"Total volume = {zone_info['Volume {m3}'].sum():.2f} m³")
else:
    print("⚠️ Could not find Zone Information table")


<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)


0  NaN        Zone Name  North Axis {deg}  Origin X-Coordinate {m}  \
0  1.0  HALL_DOWNSTAIRS               0.0                      0.0   
1  2.0       FRONT_ROOM               0.0                      0.0   
2  3.0          KITCHEN               0.0                      0.0   
3  4.0         BACKROOM               0.0                      0.0   
4  5.0        BEDROOM_3               0.0                      0.0   

0  Origin Y-Coordinate {m}  Origin Z-Coordinate {m}  \
0                      0.0                      0.0   
1                      0.0                      0.0   
2                      0.0                      0.0   
3                      0.0                      0.0   
4                      0.0                      0.0   

0  Centroid X-Coordinate {m}  Centroid Y-Coordinate {m}  \
0                       1.75                       1.24   
1                       4.09                      -0.11   
2                       4.17                       4.52   
3           

/tmp/ipykernel_30146/1791207165.py:24: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  zone_info[col] = pd.to_numeric(zone_info[col], errors="ignore")


In [2]:
# Exclude LOFT and SUBFLOOR zones
conditioned = zone_info[~zone_info["Zone Name"].str.upper().isin(["LOFT", "SUBFLOOR"])]

# Now sum only conditioned zones
total_floor_area = conditioned["Floor Area {m2}"].astype(float).sum()
total_volume     = conditioned["Volume {m3}"].astype(float).sum()

print("Conditioned floor area:", total_floor_area)
print("Conditioned volume:", total_volume)


Conditioned floor area: 85.61
Conditioned volume: 244.93


In [8]:
for df in tables[:2]:
    print(df.columns)
    if "Floor Area {m2}" in df.columns:
        zone_info = df

Index([0, 1, 2, 3], dtype='int64')
Index([0, 1], dtype='int64')


In [6]:
zone_info

In [2]:
tables

[                     0                  1  \
 0                  NaN  Total Energy [GJ]   
 1    Total Site Energy              11.73   
 2      Net Site Energy              11.73   
 3  Total Source Energy              10.25   
 4    Net Source Energy              10.25   
 
                                         2  \
 0  Energy Per Total Building Area [MJ/m2]   
 1                                   67.76   
 2                                   67.76   
 3                                   59.21   
 4                                   59.21   
 
                                               3  
 0  Energy Per Conditioned Building Area [MJ/m2]  
 1                                        137.00  
 2                                        137.00  
 3                                        119.73  
 4                                        119.73  ,
                    0                               1
 0                NaN  Site=>Source Conversion Factor
 1        Electricity        

In [ ]:
backroom = {
    (2*449*339),
    (2*357*977),
    (1*720*232),
    (1*644*989),
    (1*644*762),
}

kitchen = {
    (1*387*247),
    (1*479*729),
    (1*447*1010),
}

bedroom_3 = {
    (2*195*1000),
    (1*165*247),
    (1*165*627),
}

bedroom_1 = {
    (2*350*247),
    (2*350*777),
    (2*471*339),
    (2*471*869),
    (2*407*247),
    (2*499*869),
}

bedroom_2 = {
    (2*414*1010),
    (1*440*247),
    (1*440*637),
}

bathroom = {
    (1*387*247),
    (1*479*729),
    (1*447*1010),
}